# ЛР-02: Маршрутизация медико-эвакуационного снабжения

## Worked example: military 03

Это полностью разобранный example. Он показывает образец постановки, решения и интерпретации транспортной модели.

## 1. Постановка кейса

Открытая задача с фиктивным поставщиком, который покрывает дефицит снабжения.

### Запасы

| Поставщик | Объём |
| --- | --- |
| Склад A | 18 |
| Склад B | 24 |
| Склад C | 20 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Пункт 1 | 14 |
| Пункт 2 | 18 |
| Пункт 3 | 16 |
| Пункт 4 | 22 |

### Матрица затрат

| Откуда / Куда | Пункт 1 | Пункт 2 | Пункт 3 | Пункт 4 |
| --- | --- | --- | --- | --- |
| Склад A | 6 | 5 | 8 | 10 |
| Склад B | 5 | 4 | 6 | 7 |
| Склад C | 7 | 6 | 5 | 6 |

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog


DUMMY_SUPPLIER_NAME = "Фиктивный поставщик"
DUMMY_CONSUMER_NAME = "Фиктивный потребитель"
BALANCE_TOLERANCE = 1e-9


def make_vector_frame(values, labels, value_name):
    """Оформляет одномерный вектор как читаемую таблицу.

    Аргументы:
        values (np.ndarray): Числовой вектор для вывода.
        labels (list[str]): Подписи строк для элементов вектора.
        value_name (str): Название числового столбца.

    Возвращает:
        pd.DataFrame: Таблица с подписями и одним числовым столбцом.
    """

    return pd.DataFrame({value_name: values}, index=labels)


def balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
):
    """Балансирует открытую транспортную задачу фиктивным узлом.

    Аргументы:
        supplies (np.ndarray): Вектор запасов ``a_i``.
        demands (np.ndarray): Вектор спроса ``b_j``.
        costs (np.ndarray): Матрица затрат ``c_ij``.
        supplier_names (list[str]): Подписи строк поставщиков.
        consumer_names (list[str]): Подписи столбцов потребителей.
        dummy_cost (float): Стоимость фиктивной строки или столбца.

    Возвращает:
        tuple: Сбалансированные векторы, матрица, подписи и пояснение.
    """

    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    if balance_difference > BALANCE_TOLERANCE:
        balanced_demands = np.append(balanced_demands, balance_difference)
        balanced_consumer_names.append(DUMMY_CONSUMER_NAME)
        dummy_column = np.full(len(balanced_supplies), dummy_cost)
        balanced_costs = np.column_stack([balanced_costs, dummy_column])
        balance_note = "Добавлен фиктивный потребитель для избыточного запаса."
    elif balance_difference < -BALANCE_TOLERANCE:
        shortage = -balance_difference
        balanced_supplies = np.append(balanced_supplies, shortage)
        balanced_supplier_names.append(DUMMY_SUPPLIER_NAME)
        dummy_row = np.full(len(balanced_demands), dummy_cost)
        balanced_costs = np.vstack([balanced_costs, dummy_row])
        balance_note = "Добавлен фиктивный поставщик для дефицита."
    else:
        balance_note = "Задача уже закрытая: суммарный запас равен спросу."

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )


def make_route_labels(supplier_count, consumer_count):
    """Создает подписи для переменных развернутого вектора ``x_ij``.

    Аргументы:
        supplier_count (int): Количество строк в транспортной матрице.
        consumer_count (int): Количество столбцов в транспортной матрице.

    Возвращает:
        list[str]: Подписи в том же порядке, что и ``costs.flatten()``.
    """

    route_labels = []
    for supplier_idx in range(supplier_count):
        for consumer_idx in range(consumer_count):
            route_labels.append(f"x_{supplier_idx + 1},{consumer_idx + 1}")

    return route_labels


def make_constraint_labels(supplier_names, consumer_names):
    """Создает подписи строк для ``A_eq`` и значений ``b_eq``.

    Аргументы:
        supplier_names (list[str]): Подписи ограничений по запасам.
        consumer_names (list[str]): Подписи ограничений по спросу.

    Возвращает:
        list[str]: Подписи ограничений в том же порядке, что и ``b_eq``.
    """

    supplier_constraints = [f"запас: {name}" for name in supplier_names]
    consumer_constraints = [f"спрос: {name}" for name in consumer_names]

    return supplier_constraints + consumer_constraints


def build_transport_lp(supplies, demands, costs):
    """Собирает канонические LP-массивы для ``scipy.optimize.linprog``.

    Аргументы:
        supplies (np.ndarray): Сбалансированный вектор запасов ``a_i``.
        demands (np.ndarray): Сбалансированный вектор спроса ``b_j``.
        costs (np.ndarray): Сбалансированная матрица затрат ``c_ij``.

    Возвращает:
        dict[str, object]: Части модели ``c``, ``A_eq``, ``b_eq`` и ``bounds``.
    """

    supplier_count, consumer_count = costs.shape
    variable_count = supplier_count * consumer_count

    c = costs.flatten()
    A_eq_rows = []
    b_eq_values = []

    # Шаг 1: добавляем ограничения по запасам для каждой строки таблицы.
    for supplier_idx in range(supplier_count):
        row = np.zeros(variable_count)
        for consumer_idx in range(consumer_count):
            route_index = supplier_idx * consumer_count + consumer_idx
            row[route_index] = 1.0
        A_eq_rows.append(row)
        b_eq_values.append(supplies[supplier_idx])

    # Шаг 2: добавляем ограничения по спросу для каждого столбца таблицы.
    for consumer_idx in range(consumer_count):
        row = np.zeros(variable_count)
        for supplier_idx in range(supplier_count):
            route_index = supplier_idx * consumer_count + consumer_idx
            row[route_index] = 1.0
        A_eq_rows.append(row)
        b_eq_values.append(demands[consumer_idx])

    A_eq = np.array(A_eq_rows)
    b_eq = np.array(b_eq_values)
    bounds = [(0.0, None)] * variable_count

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
    }


def solve_transport_problem(supplies, demands, costs):
    """Решает сбалансированную транспортную задачу через ``linprog``.

    Аргументы:
        supplies (np.ndarray): Сбалансированный вектор запасов.
        demands (np.ndarray): Сбалансированный вектор спроса.
        costs (np.ndarray): Сбалансированная матрица затрат.

    Возвращает:
        tuple: ``OptimizeResult``, матрица плана и словарь LP-модели.

    Исключения:
        RuntimeError: Если HiGHS не смог решить LP-модель.
    """

    lp_model = build_transport_lp(supplies, demands, costs)
    supplier_count, consumer_count = costs.shape

    result = linprog(
        lp_model["c"],
        A_eq=lp_model["A_eq"],
        b_eq=lp_model["b_eq"],
        bounds=lp_model["bounds"],
        method="highs",
    )

    if not result.success:
        raise RuntimeError(result.message)

    plan = result.x.reshape(supplier_count, consumer_count)

    return result, plan, lp_model


def make_plan_frame(plan, supplier_names, consumer_names):
    """Оформляет оптимальную матрицу перевозок как таблицу.

    Аргументы:
        plan (np.ndarray): Оптимальная матрица перевозок ``X``.
        supplier_names (list[str]): Подписи строк.
        consumer_names (list[str]): Подписи столбцов.

    Возвращает:
        pd.DataFrame: Читаемая таблица плана перевозок.
    """

    return pd.DataFrame(plan, index=supplier_names, columns=consumer_names)


def make_used_routes_frame(plan_df, cost_df, tolerance=BALANCE_TOLERANCE):
    """Собирает таблицу всех ненулевых маршрутов оптимального плана.

    Аргументы:
        plan_df (pd.DataFrame): Таблица плана перевозок.
        cost_df (pd.DataFrame): Таблица затрат той же формы.
        tolerance (float): Малый числовой порог для ненулевых маршрутов.

    Возвращает:
        pd.DataFrame: Таблица активных маршрутов с объемами и затратами.
    """

    used_routes = []

    for supplier_name in plan_df.index:
        for consumer_name in plan_df.columns:
            volume = float(plan_df.loc[supplier_name, consumer_name])
            if volume <= tolerance:
                continue

            unit_cost = float(cost_df.loc[supplier_name, consumer_name])
            is_dummy_route = (
                supplier_name == DUMMY_SUPPLIER_NAME
                or consumer_name == DUMMY_CONSUMER_NAME
            )
            route_type = "фиктивный" if is_dummy_route else "реальный"

            used_routes.append(
                {
                    "маршрут": f"{supplier_name} -> {consumer_name}",
                    "тип": route_type,
                    "объем": round(volume, 2),
                    "тариф": unit_cost,
                    "затраты": round(volume * unit_cost, 2),
                }
            )

    return pd.DataFrame(used_routes)


def make_balance_check_frames(plan_df, supplies, demands):
    """Строит таблицы проверки баланса по строкам и столбцам.

    Аргументы:
        plan_df (pd.DataFrame): Таблица плана перевозок.
        supplies (np.ndarray): Сбалансированный вектор запасов.
        demands (np.ndarray): Сбалансированный вектор спроса.

    Возвращает:
        tuple[pd.DataFrame, pd.DataFrame]: Проверки запасов и спроса.
    """

    supply_check_df = pd.DataFrame(
        {
            "план": plan_df.sum(axis=1),
            "запас": supplies,
            "разница": plan_df.sum(axis=1) - supplies,
        },
        index=plan_df.index,
    )

    demand_check_df = pd.DataFrame(
        {
            "план": plan_df.sum(axis=0),
            "спрос": demands,
            "разница": plan_df.sum(axis=0) - demands,
        },
        index=plan_df.columns,
    )

    return supply_check_df, demand_check_df


In [2]:
# Шаг 1: записываем данные ровно как в условии задачи.
supplier_names = [
    'Склад A',
    'Склад B',
    'Склад C',
]
consumer_names = [
    'Пункт 1',
    'Пункт 2',
    'Пункт 3',
    'Пункт 4',
]

supplies = np.array(
    [
        18,
        24,
        20,
    ],
    dtype=float,
)

demands = np.array(
    [
        14,
        18,
        16,
        22,
    ],
    dtype=float,
)

costs = np.array(
    [
        [6, 5, 8, 10],
        [5, 4, 6, 7],
        [7, 6, 5, 6],
    ],
    dtype=float,
)

# Шаг 2: выводим векторы и матрицу затрат в читаемом табличном виде.
supply_df = make_vector_frame(supplies, supplier_names, "запас")
demand_df = make_vector_frame(demands, consumer_names, "спрос")
cost_df_raw = pd.DataFrame(costs, index=supplier_names, columns=consumer_names)

print("Запасы поставщиков a_i:")
display(supply_df)

print("Спрос потребителей b_j:")
display(demand_df)

print("Матрица затрат c_ij:")
display(cost_df_raw)

# Шаг 3: проверяем баланс и при необходимости закрываем модель.
(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
    balanced_supplier_names,
    balanced_consumer_names,
    balance_note,
) = balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
)

print(balance_note)
print("sum supply =", balanced_supplies.sum())
print("sum demand =", balanced_demands.sum())

assert np.allclose(
    balanced_supplies.sum(),
    balanced_demands.sum(),
), "После балансировки сумма запасов должна равняться сумме спроса."

# Шаг 4: решаем каноническую LP-модель min c^T x, A_eq x = b_eq, x >= 0.
result, plan, lp_model = solve_transport_problem(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
)

route_labels = make_route_labels(
    len(balanced_supplier_names),
    len(balanced_consumer_names),
)
constraint_labels = make_constraint_labels(
    balanced_supplier_names,
    balanced_consumer_names,
)

c_df = pd.DataFrame(
    {"переменная": route_labels, "стоимость c": lp_model["c"]}
)
A_eq_df = pd.DataFrame(lp_model["A_eq"], columns=route_labels)
b_eq_df = pd.DataFrame(
    {"ограничение": constraint_labels, "b_eq": lp_model["b_eq"]}
)

print("Вектор цели c = costs.flatten():")
display(c_df)

print("Матрица ограничений A_eq:")
display(A_eq_df)

print("Вектор правых частей b_eq:")
display(b_eq_df)

plan_df = make_plan_frame(
    plan,
    balanced_supplier_names,
    balanced_consumer_names,
)
cost_df = pd.DataFrame(
    balanced_costs,
    index=balanced_supplier_names,
    columns=balanced_consumer_names,
)

print("Оптимальная стоимость:", round(result.fun, 2))
print("План перевозок X:")
display(plan_df)

print("Матрица затрат после балансировки:")
display(cost_df)


Запасы поставщиков a_i:


,запас
Склад A,18.0
Склад B,24.0
Склад C,20.0


Спрос потребителей b_j:


,спрос
Пункт 1,14.0
Пункт 2,18.0
Пункт 3,16.0
Пункт 4,22.0


Матрица затрат c_ij:


...

,ограничение,b_eq
0,запас: Склад A,18.0
1,запас: Склад B,24.0
2,запас: Склад C,20.0
3,запас: Фиктивный поставщик,8.0
4,спрос: Пункт 1,14.0
5,спрос: Пункт 2,18.0
6,спрос: Пункт 3,16.0
7,спрос: Пункт 4,22.0


Оптимальная стоимость: 334.0
План перевозок X:


,Пункт 1,Пункт 2,Пункт 3,Пункт 4
Склад A,0.0,18.0,0.0,0.0
Склад B,14.0,-0.0,0.0,10.0
Склад C,0.0,0.0,16.0,4.0
Фиктивный поставщик,0.0,0.0,0.0,8.0


Матрица затрат после балансировки:


,Пункт 1,Пункт 2,Пункт 3,Пункт 4
Склад A,6.0,5.0,8.0,10.0
Склад B,5.0,4.0,6.0,7.0
Склад C,7.0,6.0,5.0,6.0
Фиктивный поставщик,0.0,0.0,0.0,0.0


In [3]:
# Шаг 5: перечисляем активные маршруты и проверяем допустимость плана.
used_routes_df = make_used_routes_frame(plan_df, cost_df)

supply_check_df, demand_check_df = make_balance_check_frames(
    plan_df,
    balanced_supplies,
    balanced_demands,
)

print("Использованные маршруты:")
display(used_routes_df)

print("Проверка баланса по поставщикам:")
display(supply_check_df)

print("Проверка баланса по потребителям:")
display(demand_check_df)

assert np.allclose(
    supply_check_df["план"],
    supply_check_df["запас"],
), "Суммы по строкам должны совпадать с запасами."

assert np.allclose(
    demand_check_df["план"],
    demand_check_df["спрос"],
), "Суммы по столбцам должны совпадать со спросом."

assert np.allclose(
    float((plan_df * cost_df).to_numpy().sum()),
    result.fun,
), "Стоимость по таблицам должна совпадать с result.fun."


Использованные маршруты:


,маршрут,тип,объем,тариф,затраты
0,Склад A -> Пункт 2,реальный,18.0,5.0,90.0
1,Склад B -> Пункт 1,реальный,14.0,5.0,70.0
2,Склад B -> Пункт 4,реальный,10.0,7.0,70.0
3,Склад C -> Пункт 3,реальный,16.0,5.0,80.0
4,Склад C -> Пункт 4,реальный,4.0,6.0,24.0
5,Фиктивный поставщик -> Пункт 4,фиктивный,8.0,0.0,0.0


Проверка баланса по поставщикам:


,план,запас,разница
Склад A,18.0,18.0,0.0
Склад B,24.0,24.0,0.0
Склад C,20.0,20.0,0.0
Фиктивный поставщик,8.0,8.0,0.0


Проверка баланса по потребителям:


,план,спрос,разница
Пункт 1,14.0,14.0,0.0
Пункт 2,18.0,18.0,0.0
Пункт 3,16.0,16.0,0.0
Пункт 4,22.0,22.0,0.0


## 2. Что важно проговорить в выводе

- сначала проверьте, закрытая задача или открытая;
- после балансировки объясните смысл фиктивного узла, если он появился;
- в LP-форме явно покажите `c`, `A_eq`, `b_eq`, `bounds`;
- после решения сверяйте суммы по строкам и столбцам;
- отдельно перечисляйте ненулевые реальные маршруты;
- фиктивные перевозки не являются реальной доставкой: это резерв или дефицит.

Для отчёта недостаточно назвать только оптимальную стоимость. Важно показать,
почему план допустим и как его читать на языке предметной области.
